# 03 - Ewaluacja modeli na zbiorze zakłóconym ImageNet-1k

**Praca magisterska:** *„Analiza wpływu zakłóceń i perturbacji obrazów na stabilność reprezentacji generowanych przez głębokie sieci neuronowe, z uwzględnieniem klas obiektów"*

---

Notebook przeprowadza pełną ewaluację sześciu pretrenowanych modeli na zbiorze
walidacyjnym ImageNet-1k poddanym **15 typom zakłóceń** w **5 poziomach intensywności**,
zgodnie z protokołem ImageNet-C [Hendrycks & Dietterich, 2019].

Łączna liczba ewaluacji per model: 50 000 obrazów x 76 wariantów = **3 800 000**.

| Model | Rodzina | Parametry | Wagi |
|---|---|---|---|
| ResNet-50  | CNN | 25.6 M | IMAGENET1K_V2 |
| ResNet-101 | CNN | 44.5 M | IMAGENET1K_V2 |
| ResNet-152 | CNN | 60.2 M | IMAGENET1K_V2 |
| ViT-B/32   | ViT | 88.2 M | IMAGENET1K_V1 |
| ViT-B/16   | ViT | 86.6 M | IMAGENET1K_V1 |
| ViT-L/16   | ViT | 307 M  | IMAGENET1K_V1 |

Dla każdej kombinacji (obraz x zakłócenie x severity) zapisywane są: top-1 i top-5 accuracy,
prawdopodobieństwo przyznane prawdziwej klasie, predykcja top-1 z pewnością oraz
pełna piątka top-5 z prawdopodobieństwami.

Wyniki trafiają do `mgr_266484/corruped/{model}/per_image.csv`.



## 1. Środowisko obliczeniowe

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q imagecorruptions datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 33.6 MB/s eta 0:00:00


In [ ]:
import os, json, random, time, gc
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from datasets import load_dataset, VerificationMode
from huggingface_hub import login


np.float_   = np.float64
np.int_     = np.int64
np.complex_ = np.complex128
np.bool8    = np.bool_

import skimage.filters as _sk_filters
import imagecorruptions.corruptions as _ic_corruptions

if not getattr(_sk_filters.gaussian, '_patched', False):
    _orig = _sk_filters.gaussian
    def _compat(*a, **kw):
        if 'multichannel' in kw:
            mc = kw.pop('multichannel')
            kw.setdefault('channel_axis', -1 if mc else None)
        return _orig(*a, **kw)
    _compat._patched = True
    _sk_filters.gaussian = _ic_corruptions.gaussian = _compat

from imagecorruptions import corrupt, get_corruption_names

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU:    {torch.cuda.get_device_name(0)}')
    print(f'VRAM:   {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

Device: cuda
GPU:    Tesla T4
VRAM:   15.6 GB


## 2. Konfiguracja

Wszystkie ścieżki, modele i parametry zakłóceń zdefiniowane w jednym miejscu.
Aby dodać nowy model wystarczy dopisać jeden wpis do `MODELS` i `BATCH_SIZES`.


In [ ]:
ROOT         = '/content/drive/MyDrive/mgr_266484'
DATA_DIR     = f'{ROOT}/data'
BASELINE_DIR = f'{ROOT}/baseline'
RESULTS_DIR  = f'{ROOT}/corrupted'
FIGURES_DIR  = f'{ROOT}/figures'

for d in [RESULTS_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

MODELS = {
    'resnet50':  lambda: models.resnet50 (weights=models.ResNet50_Weights.IMAGENET1K_V2),
    'resnet101': lambda: models.resnet101(weights=models.ResNet101_Weights.IMAGENET1K_V2),
    'resnet152': lambda: models.resnet152(weights=models.ResNet152_Weights.IMAGENET1K_V2),
    'vit_b_32':  lambda: models.vit_b_32 (weights=models.ViT_B_32_Weights.IMAGENET1K_V1),
    'vit_b_16':  lambda: models.vit_b_16 (weights=models.ViT_B_16_Weights.IMAGENET1K_V1),
    'vit_l_16':  lambda: models.vit_l_16 (weights=models.ViT_L_16_Weights.IMAGENET1K_V1),
}

BATCH_SIZES = {
    'resnet50': 256, 'resnet101': 128, 'resnet152': 128,
    'vit_b_32': 128, 'vit_b_16':   64, 'vit_l_16':   32,
}

CORRUPTIONS = {
    'noise':   ['gaussian_noise', 'shot_noise', 'impulse_noise'],
    'blur':    ['defocus_blur', 'glass_blur', 'motion_blur', 'zoom_blur'],
    'weather': ['snow', 'frost', 'fog', 'brightness'],
    'digital': ['contrast', 'elastic_transform', 'pixelate', 'jpeg_compression'],
}
SEVERITIES        = [1, 2, 3, 4, 5]
ALL_CORRUPTIONS   = [c for lst in CORRUPTIONS.values() for c in lst]
CORRUPTION_TO_CAT = {c: cat for cat, lst in CORRUPTIONS.items() for c in lst}
ALL_VARIANTS      = [('clean', 0)] + [(c, s) for c in ALL_CORRUPTIONS for s in SEVERITIES]

CHECKPOINT_EVERY = 1
NUM_WORKERS      = 4

print(f'Modeli:              {len(MODELS)}')
print(f'Wariantów:           {len(ALL_VARIANTS)}  (1 clean + {len(ALL_CORRUPTIONS)*len(SEVERITIES)} zakłóconych)')
print(f'Ewaluacji per model: {50000 * len(ALL_VARIANTS):,}')

Modeli:              6
Wariantów:           76  (1 clean + 75 zakłóconych)
Ewaluacji per model: 3,800,000


## 3. Wczytanie zbioru i mapowań etykiet

Mapowania identyczne jak w notebookach 01 i 02, aby pliki wynikowe
były złączalne kluczem `filename`.


In [ ]:
label_map      = pd.read_csv(f'{DATA_DIR}/label_map.csv')
idx_to_wnid    = dict(zip(label_map['imagenet1k_label'], label_map['wnid']))
inet1k_to_name = {}
with open(f'{DATA_DIR}/imagenet_class_index.json') as f:
    for k, v in json.load(f).items():
        inet1k_to_name[int(k)] = v[1]

os.environ['HF_DATASETS_CACHE'] = f'{ROOT}/hf_cache'

ds_val = load_dataset(
    'ILSVRC/imagenet-1k',
    split='validation',
    data_files={'validation': 'data/validation-*'},
    verification_mode=VerificationMode.NO_CHECKS
)
print(f'Zbiór:    {len(ds_val):,} obrazów')
print(f'Mapowań:  {len(label_map)} klas')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/87.6k [00:00<?, ?B/s]

data/validation-00000-of-00014.parquet:   0%|          | 0.00/480M [00:00<?, ?B/s]

data/validation-00001-of-00014.parquet:   0%|          | 0.00/485M [00:00<?, ?B/s]

data/validation-00002-of-00014.parquet:   0%|          | 0.00/471M [00:00<?, ?B/s]

...

data/validation-00011-of-00014.parquet:   0%|          | 0.00/472M [00:00<?, ?B/s]

data/validation-00012-of-00014.parquet:   0%|          | 0.00/493M [00:00<?, ?B/s]

data/validation-00013-of-00014.parquet:   0%|          | 0.00/484M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Zbiór:    50,000 obrazów
Mapowań:  1000 klas


## 4. Klasa `CorruptedImageNetDataset`

Deterministyczny generator zakłóconych obrazów - `corrupt(img, name, severity)` daje zawsze identyczny wynik dla identycznego wejścia.

Preprocessing: Resize 256 → CenterCrop 224 → uint8 → corrupt → ToTensor → Normalize.
Normalizacja wykonywana jest po zakłóceniu, zgodnie z protokołem ImageNet-C.


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def to_uint8_224(pil_img):
    """Resize (krótszy bok → 256) + CenterCrop 224 → np.uint8 (H,W,3)."""
    img = pil_img.convert('RGB')
    w, h = img.size
    s = 256
    img = img.resize((s, int(s*h/w)) if w < h else (int(s*w/h), s), Image.BILINEAR)
    w2, h2 = img.size
    img = img.crop(((w2-224)//2, (h2-224)//2, (w2-224)//2+224, (h2-224)//2+224))
    return np.asarray(img, dtype=np.uint8)

class CorruptedImageNetDataset(Dataset):
    _mean = torch.tensor(IMAGENET_MEAN).view(3,1,1)
    _std  = torch.tensor(IMAGENET_STD).view(3,1,1)

    def __init__(self, hf_ds, corruption='clean', severity=1):
        self.ds         = hf_ds
        self.corruption = corruption
        self.severity   = severity
        self._to_tensor = transforms.ToTensor()

    def __len__(self): return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]
        lbl  = int(item['label'])
        arr  = to_uint8_224(item['image'])
        if self.corruption != 'clean':
            arr = corrupt(arr, corruption_name=self.corruption, severity=self.severity)
            if arr.dtype != np.uint8:
                arr = np.clip(arr, 0, 255).astype(np.uint8)
        tensor = (self._to_tensor(Image.fromarray(arr)) - self._mean) / self._std
        return tensor, lbl, idx_to_wnid[lbl], f'inet1k_val_{idx:06d}'

## 5. Patch wydajnościowy: glass_blur

Oryginalna implementacja `glass_blur` w `imagecorruptions` jest silnie CPU-bound
(~25+ min na 50 000 obrazów dla jednego wariantu severity). Poniższy patch zastępuje
ją wektoryzowaną wersją NumPy, osiągając ~10× przyspieszenie przy zachowaniu
identycznej semantyki zakłócenia.


In [ ]:
def _fast_glass_blur(x, severity=1):
    c = [(0.7,1,2),(0.9,2,1),(1,2,3),(1.1,3,2),(1.5,4,2)][severity-1]
    sigma, max_delta, n_iter = c

    arr = np.asarray(x, dtype=np.float32) / 255.0
    arr = _sk_filters.gaussian(arr, sigma=sigma, channel_axis=-1)
    arr = (arr * 255).astype(np.uint8)

    H, W = arr.shape[:2]
    rng  = np.random.RandomState()
    for _ in range(n_iter):
        dy = rng.randint(-max_delta, max_delta+1, size=(H,W))
        dx = rng.randint(-max_delta, max_delta+1, size=(H,W))
        ii = np.clip(np.arange(H)[:,None] + dy, 0, H-1)
        jj = np.clip(np.arange(W)[None,:] + dx, 0, W-1)
        arr = arr[ii, jj]

    arr = _sk_filters.gaussian(arr.astype(np.float32)/255.0, sigma=sigma, channel_axis=-1)
    return np.clip(arr * 255, 0, 255).astype(np.uint8)

_ic_corruptions.glass_blur = _fast_glass_blur

import imagecorruptions as _imagecorruptions
if hasattr(_imagecorruptions, 'corruption_dict'):
    _imagecorruptions.corruption_dict['glass_blur'] = _fast_glass_blur

assert _ic_corruptions.glass_blur is _fast_glass_blur, 'Patch NIE zainstalowany!'

## 6. Funkcja ewaluacji z checkpointami

Rozmiary batchy dobrane pod Colab Pro (GPU A100/V100):
ResNet-50: 256, ResNet-101/152: 128, ViT-B/32: 128, ViT-B/16: 64, ViT-L/16: 32.


In [ ]:
@torch.no_grad()
def evaluate_variant(model, corruption, severity, batch_size):
    ds     = CorruptedImageNetDataset(ds_val, corruption, severity)
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
    category = CORRUPTION_TO_CAT.get(corruption, 'clean')
    records  = []

    for imgs, labels, wnids, fnames in loader:
        imgs     = imgs.to(DEVICE, non_blocking=True)
        probs_t  = torch.softmax(model(imgs), dim=1)
        probs_np = probs_t.cpu().numpy()
        top5_p, top5_i = probs_t.topk(5, dim=1)
        top5_i = top5_i.cpu().numpy()
        top5_p = top5_p.cpu().numpy()

        for j in range(len(imgs)):
            lbl   = int(labels[j])
            top5i = top5_i[j]
            top5p = top5_p[j]
            records.append({
                'filename':        fnames[j],
                'wnid':            wnids[j],
                'true_idx':        lbl,
                'corruption':      corruption,
                'category':        category,
                'severity':        severity,
                'top1_correct':    int(top5i[0] == lbl),
                'top5_correct':    int(lbl in top5i),
                'true_class_prob': float(probs_np[j, lbl]),
                'pred_top1':       int(top5i[0]),
                'pred_top1_prob':  float(top5p[0]),
                'top5_preds':      ','.join(map(str, top5i.tolist())),
                'top5_probs':      ','.join(f'{p:.5f}' for p in top5p.tolist()),
            })
    return records


def run_evaluation(model_name):
    """Pełna ewaluacja modelu na wszystkich wariantach z obsługą wznawiania."""
    model_dir = f'{RESULTS_DIR}/{model_name}'
    os.makedirs(model_dir, exist_ok=True)
    out_csv = f'{model_dir}/per_image.csv'
    ckpt_f  = f'{model_dir}/.checkpoint.json'

    done = set()
    if os.path.exists(ckpt_f):
        with open(ckpt_f) as f:
            done = set(map(tuple, json.load(f)))
        print(f'{model_name}: wznawianie - {len(done)}/{len(ALL_VARIANTS)} wariantów ukończonych')

    remaining = [v for v in ALL_VARIANTS if v not in done]
    if not remaining:
        print(f'{model_name}: wszystkie warianty ukończone.')
        return

    model = MODELS[model_name]().eval().to(DEVICE)
    bs    = BATCH_SIZES[model_name]
    buf   = []
    t_start = time.time()

    for i, (corruption, severity) in enumerate(tqdm(remaining, desc=model_name)):
        t0  = time.time()
        buf += evaluate_variant(model, corruption, severity, bs)
        done.add((corruption, severity))

        if (i + 1) % CHECKPOINT_EVERY == 0 or i == len(remaining) - 1:
            df = pd.DataFrame(buf)
            write_header = not os.path.exists(out_csv)
            df.to_csv(out_csv, mode='a', header=write_header, index=False)
            buf = []
            with open(ckpt_f, 'w') as f:
                json.dump([list(v) for v in done], f)
            tqdm.write(f'  [{model_name}] {len(done)}/{len(ALL_VARIANTS)} wariantów | '
                       f'ostatni: {time.time()-t0:.1f}s')
            gc.collect(); torch.cuda.empty_cache()

    elapsed = time.time() - t_start
    del model; gc.collect(); torch.cuda.empty_cache()
    print(f'{model_name}: gotowe - {out_csv}  ({elapsed/60:.1f} min)')
    if os.path.exists(ckpt_f):
        os.remove(ckpt_f)

In [ ]:
from imagecorruptions import corruption_dict
corruption_dict['glass_blur'] = _fast_glass_blur
print(corruption_dict['glass_blur'].__name__)

_fast_glass_blur


## 7. Weryfikacja pipeline'u

Przed uruchomieniem wielogodzinnej ewaluacji weryfikujemy pipeline na próbce:
1. Czy accuracy na czystych obrazach jest zgodne z baseline z notebooka 01?
2. Czy zakłócenia powodują istotny spadek accuracy przy severity 5?
3. Czy format wyjściowych wierszy jest poprawny?


In [ ]:
print('=== Weryfikacja pipeline\'u ===\n')

baseline_top1 = {}
if os.path.exists(f'{BASELINE_DIR}/baseline_summary.csv'):
    bs_df = pd.read_csv(f'{BASELINE_DIR}/baseline_summary.csv')
    baseline_top1 = dict(zip(bs_df['model'], bs_df['top1_acc'] / 100))
    print('Baseline z notebooka 01:')
    print(bs_df.to_string(index=False))
    print()

for model_name in MODELS:
    model = MODELS[model_name]().eval().to(DEVICE)

    # Test 1: czyste obrazy
    rec_clean = evaluate_variant(model, 'clean', 0, BATCH_SIZES[model_name])
    df_clean  = pd.DataFrame(rec_clean)
    top1_clean = df_clean['top1_correct'].mean()

    # Test 2: gaussian_noise severity 5
    rec_noisy = evaluate_variant(model, 'gaussian_noise', 5, BATCH_SIZES[model_name])
    df_noisy  = pd.DataFrame(rec_noisy)
    top1_noisy = df_noisy['top1_correct'].mean()
    drop = top1_clean - top1_noisy

    delta_vs_bl = abs(top1_clean - baseline_top1.get(model_name, top1_clean))
    status = 'OK' if delta_vs_bl < 0.005 else f'UWAGA: delta={delta_vs_bl:.4f}'

    print(f'{model_name:<12}  clean={top1_clean:.4f}  noise_s5={top1_noisy:.4f}  '
          f'drop={drop:+.4f}  [{status}]')

    assert drop > 0.05, f'Zbyt mały spadek accuracy dla {model_name} - sprawdź pipeline.'

    del model; gc.collect(); torch.cuda.empty_cache()

print('\nWeryfikacja OK.')
print('\nPrzykładowy wiersz wyjściowy:')
for k, v in rec_clean[0].items():
    print(f'  {k:<20} {repr(v)[:60]}')

## 8. Pełna ewaluacja

Modele uruchamiane sekwencyjnie, jeden model na raz.

In [ ]:
for model_name in ['resnet50']:
    print('=' * 60)
    print(f'EWALUACJA: {model_name.upper()}')
    print('=' * 60)
    print(f'Wariantów:   {len(ALL_VARIANTS)}')
    print(f'Obrazów:     {len(ds_val):,}')
    print(f'Wierszy:     {len(ALL_VARIANTS) * len(ds_val):,}')
    print(f'Batch size:  {BATCH_SIZES[model_name]}')
    print(f'Wyjście:     {RESULTS_DIR}/{model_name}/per_image.csv')
    print()
    run_evaluation(model_name)
    print()

print('\n' + '='*60)
print('WSZYSTKIE MODELE UKOŃCZONE.')
print('='*60)

In [ ]:
for model_name in ['vit_b_16']:
    print('=' * 60)
    print(f'EWALUACJA: {model_name.upper()}')
    print('=' * 60)
    print(f'Wariantów:   {len(ALL_VARIANTS)}')
    print(f'Obrazów:     {len(ds_val):,}')
    print(f'Wierszy:     {len(ALL_VARIANTS) * len(ds_val):,}')
    print(f'Batch size:  {BATCH_SIZES[model_name]}')
    print(f'Wyjście:     {RESULTS_DIR}/{model_name}/per_image.csv')
    print()
    run_evaluation(model_name)
    print()

In [ ]:
path = '/content/drive/MyDrive/mgr_266484/corrupted/vit_b_16/per_image.csv'
n = sum(1 for _ in open(path)) - 1
print(f'Wierszy: {n:,}')
print(f'Oczekiwano: {3_800_000:,}')

In [ ]:
for model_name in ['vit_b_32']:
    print('=' * 60)
    print(f'EWALUACJA: {model_name.upper()}')
    print('=' * 60)
    print(f'Wariantów:   {len(ALL_VARIANTS)}')
    print(f'Obrazów:     {len(ds_val):,}')
    print(f'Wierszy:     {len(ALL_VARIANTS) * len(ds_val):,}')
    print(f'Batch size:  {BATCH_SIZES[model_name]}')
    print(f'Wyjście:     {RESULTS_DIR}/{model_name}/per_image.csv')
    print()
    run_evaluation(model_name)
    print()

In [ ]:
path = '/content/drive/MyDrive/mgr_266484/corrupted/vit_b_32/per_image.csv'
n = sum(1 for _ in open(path)) - 1
print(f'Wierszy: {n:,}')
print(f'Oczekiwano: {3_800_000:,}')

In [ ]:
import pandas as pd, os, json

model_name = 'resnet50'
model_dir  = f'{RESULTS_DIR}/{model_name}'
out_csv    = f'{model_dir}/per_image.csv'
ckpt_f     = f'{model_dir}/.checkpoint.json'

df  = pd.read_csv(out_csv)
cnt = df.groupby(['corruption', 'severity']).size()
N   = int(cnt.max())
print('Pełny wariant =', N, 'wierszy')

complete = {(c, int(s)) for (c, s), k in cnt.items() if k == N}
partial  = {(c, int(s)) for (c, s), k in cnt.items() if k != N}

if partial:
    print('Niekompletne, usuwam i doliczę:', sorted(partial))
    df['_k'] = df['corruption'] + '|' + df['severity'].astype(int).astype(str)
    bad = {f'{c}|{s}' for (c, s) in partial}
    df[~df['_k'].isin(bad)].drop(columns='_k').to_csv(out_csv, index=False)

with open(ckpt_f, 'w') as f:
    json.dump([list(v) for v in complete], f)

missing = [v for v in ALL_VARIANTS if v not in complete]
print(f'Kompletnych: {len(complete)}/{len(ALL_VARIANTS)}  |  do doliczenia: {len(missing)}')
print('Brakujące:', missing)

In [ ]:
for model_name in ['resnet50']:
    print('=' * 60)
    print(f'EWALUACJA: {model_name.upper()}')
    print('=' * 60)
    print(f'Wariantów:   {len(ALL_VARIANTS)}')
    print(f'Obrazów:     {len(ds_val):,}')
    print(f'Wierszy:     {len(ALL_VARIANTS) * len(ds_val):,}')
    print(f'Batch size:  {BATCH_SIZES[model_name]}')
    print(f'Wyjście:     {RESULTS_DIR}/{model_name}/per_image.csv')
    print()
    run_evaluation(model_name)
    print()

print('\n' + '='*60)
print('WSZYSTKIE MODELE UKOŃCZONE.')
print('='*60)

EWALUACJA: RESNET50
Wariantów:   76
Obrazów:     50,000
Wierszy:     3,800,000
Batch size:  256
Wyjście:     /content/drive/MyDrive/mgr_266484/corrupted/resnet50/per_image.csv

resnet50: wznawianie - 62/76 wariantów ukończonych
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 224MB/s]


resnet50:   0%|          | 0/14 [00:00<?, ?it/s]

  [resnet50] 63/76 wariantów | ostatni: 241.2s
  [resnet50] 64/76 wariantów | ostatni: 399.1s
  [resnet50] 65/76 wariantów | ostatni: 397.6s
  [resnet50] 66/76 wariantów | ostatni: 398.2s
  [resnet50] 67/76 wariantów | ostatni: 93.1s
  [resnet50] 68/76 wariantów | ostatni: 93.5s
  [resnet50] 69/76 wariantów | ostatni: 92.9s
  [resnet50] 70/76 wariantów | ostatni: 92.4s
  [resnet50] 71/76 wariantów | ostatni: 91.7s
  [resnet50] 72/76 wariantów | ostatni: 95.0s
  [resnet50] 73/76 wariantów | ostatni: 94.3s
  [resnet50] 74/76 wariantów | ostatni: 94.7s
  [resnet50] 75/76 wariantów | ostatni: 94.4s
  [resnet50] 76/76 wariantów | ostatni: 94.1s
resnet50: gotowe - /content/drive/MyDrive/mgr_266484/corrupted/resnet50/per_image.csv  (39.6 min)


WSZYSTKIE MODELE UKOŃCZONE.


In [ ]:
import pandas as pd, os, json
model_dir = f'{RESULTS_DIR}/resnet50'
out_csv, ckpt_f = f'{model_dir}/per_image.csv', f'{model_dir}/.checkpoint.json'

df  = pd.read_csv(out_csv, low_memory=False)
cnt = df.groupby(['corruption','severity']).size()

# wariant kompletny tylko jeśli ma 50000 wierszy
complete = {(c, int(s)) for (c, s), k in cnt.items() if k == 50000}
partial  = {(c, int(s)) for (c, s), k in cnt.items() if k != 50000}

# usuń niedokończone warianty z CSV, żeby doliczyć je czysto (bez duplikatów)
if partial:
    df['_k'] = df.corruption + '|' + df.severity.astype(int).astype(str)
    df[~df._k.isin({f'{c}|{s}' for c, s in partial})].drop(columns='_k').to_csv(out_csv, index=False)
    print('Usunięto niekompletne:', sorted(partial))

with open(ckpt_f, 'w') as f:
    json.dump([list(v) for v in complete], f)
print('Do doliczenia:', [v for v in ALL_VARIANTS if v not in complete])

Usunięto niekompletne: [('glass_blur', 5), ('jpeg_compression', 5)]
Do doliczenia: [('glass_blur', 5), ('jpeg_compression', 5)]


In [ ]:
run_evaluation('resnet50')

resnet50: wznawianie - 74/76 wariantów ukończonych


resnet50:   0%|          | 0/2 [00:00<?, ?it/s]

  [resnet50] 75/76 wariantów | ostatni: 251.7s
  [resnet50] 76/76 wariantów | ostatni: 98.3s
resnet50: gotowe - /content/drive/MyDrive/mgr_266484/corrupted/resnet50/per_image.csv  (5.8 min)


In [ ]:
run_evaluation('resnet50')

resnet50: wznawianie - 75/76 wariantów ukończonych
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 240MB/s]


resnet50:   0%|          | 0/1 [00:00<?, ?it/s]

  [resnet50] 76/76 wariantów | ostatni: 251.7s
resnet50: gotowe - /content/drive/MyDrive/mgr_266484/corrupted/resnet50/per_image.csv  (4.2 min)


In [ ]:
import pandas as pd
cnt = pd.read_csv(f'{RESULTS_DIR}/resnet50/per_image.csv', low_memory=False)\
        .groupby(['corruption','severity']).size()
print('Wariantów:', cnt.shape[0], '/ 76')
print('Niekompletne:', cnt[cnt != 50000].to_string() or 'wszystkie po 50000')

masz = set(map(tuple, cnt.reset_index()[['corruption','severity']].values))
ALL_C = ['gaussian_noise','shot_noise','impulse_noise','defocus_blur','glass_blur',
         'motion_blur','zoom_blur','snow','frost','fog','brightness','contrast',
         'elastic_transform','pixelate','jpeg_compression']
brak = [(c, s) for c in ALL_C for s in [1,2,3,4,5] if (c, float(s)) not in masz]
if ('clean', 0.0) not in masz: brak.append(('clean', 0))
print('Brakujące całkiem:', brak)

Wariantów: 76 / 76
Niekompletne: Series([], )
Brakujące całkiem: []


In [ ]:
for model_name in ['resnet152']:
    print('=' * 60)
    print(f'EWALUACJA: {model_name.upper()}')
    print('=' * 60)
    print(f'Wariantów:   {len(ALL_VARIANTS)}')
    print(f'Obrazów:     {len(ds_val):,}')
    print(f'Wierszy:     {len(ALL_VARIANTS) * len(ds_val):,}')
    print(f'Batch size:  {BATCH_SIZES[model_name]}')
    print(f'Wyjście:     {RESULTS_DIR}/{model_name}/per_image.csv')
    print()
    run_evaluation(model_name)
    print()

print('\n' + '='*60)
print('WSZYSTKIE MODELE UKOŃCZONE.')
print('='*60)

EWALUACJA: RESNET152
Wariantów:   76
Obrazów:     50,000
Wierszy:     3,800,000
Batch size:  128
Wyjście:     /content/drive/MyDrive/mgr_266484/corrupted/resnet152/per_image.csv

Downloading: "https://download.pytorch.org/models/resnet152-f82ba261.pth" to /root/.cache/torch/hub/checkpoints/resnet152-f82ba261.pth


100%|██████████| 230M/230M [00:00<00:00, 245MB/s]


resnet152:   0%|          | 0/76 [00:00<?, ?it/s]

  [resnet152] 1/76 wariantów | ostatni: 167.1s
  [resnet152] 2/76 wariantów | ostatni: 167.3s
  [resnet152] 3/76 wariantów | ostatni: 167.0s
  [resnet152] 4/76 wariantów | ostatni: 167.0s
  [resnet152] 5/76 wariantów | ostatni: 167.0s
  [resnet152] 6/76 wariantów | ostatni: 167.0s
  [resnet152] 7/76 wariantów | ostatni: 283.9s
  [resnet152] 8/76 wariantów | ostatni: 297.1s
  [resnet152] 9/76 wariantów | ostatni: 288.2s
  [resnet152] 10/76 wariantów | ostatni: 237.4s
  [resnet152] 11/76 wariantów | ostatni: 217.0s
  [resnet152] 12/76 wariantów | ostatni: 166.9s
  [resnet152] 13/76 wariantów | ostatni: 166.9s
  [resnet152] 14/76 wariantów | ostatni: 166.8s
  [resnet152] 15/76 wariantów | ostatni: 166.9s
  [resnet152] 16/76 wariantów | ostatni: 166.9s
  [resnet152] 17/76 wariantów | ostatni: 217.8s
  [resnet152] 18/76 wariantów | ostatni: 217.2s
  [resnet152] 19/76 wariantów | ostatni: 218.7s
  [resnet152] 20/76 wariantów | ostatni: 219.6s
  [resnet152] 21/76 wariantów | ostatni: 233.5s
 

## 9. Weryfikacja kompletności wyników

In [ ]:
expected = len(ds_val) * len(ALL_VARIANTS)
print(f'Oczekiwana liczba wierszy per model: {expected:,}\n')

summary_rows = []
for model_name in MODELS:
    path = f'{RESULTS_DIR}/{model_name}/per_image.csv'
    if not os.path.exists(path):
        print(f'{model_name:<12} - brak pliku')
        continue
    df = pd.read_csv(path)
    n          = len(df)
    n_variants = df.groupby(['corruption','severity']).ngroups
    size_mb    = os.path.getsize(path) / 1024**2
    status     = 'OK' if n == expected else f'oczekiwano {expected:,}'

    clean_top1 = df[df['corruption']=='clean']['top1_correct'].mean()
    corr_top1  = df[df['corruption']!='clean']['top1_correct'].mean()

    print(f'{model_name:<12}  {n:>10,} wierszy  {n_variants:>3} wariantów  '
          f'{size_mb:>6.1f} MB  {status}')
    print(f'              clean top-1={clean_top1:.4f}  corrupted top-1={corr_top1:.4f}')

    summary_rows.append({'model': model_name,
                         'clean_top1':     round(clean_top1*100, 2),
                         'corrupted_top1': round(corr_top1*100,  2),
                         'mCE':            round((1-corr_top1)*100, 2)})

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(f'{RESULTS_DIR}/eval_summary.csv', index=False)
print(f'\nPodsumowanie zapisano: {RESULTS_DIR}/eval_summary.csv')
print()
print(summary_df.to_string(index=False))

## 10. Agregowane tabele wynikowe

Cztery tabele CSV przyspieszające pracę w notebooku 04 - wszystkie obliczane
z plików per_image.csv, nie wymagają ponownego uruchamiania modeli.


In [ ]:
dfs = {}
for model_name in MODELS:
    path = f'{RESULTS_DIR}/{model_name}/per_image.csv'
    if os.path.exists(path):
        dfs[model_name] = pd.read_csv(path).assign(model=model_name)

df_all = pd.concat(dfs.values(), ignore_index=True)
print(f'Łącznie wierszy (wszystkie modele): {len(df_all):,}')

In [ ]:
# Tabela 1: per (model, corruption, severity)
agg_corr_sev = (df_all.groupby(['model','corruption','category','severity'])
                .agg(n              = ('filename',       'count'),
                     top1_acc       = ('top1_correct',   'mean'),
                     top5_acc       = ('top5_correct',   'mean'),
                     mean_true_prob = ('true_class_prob','mean'),
                     mean_conf      = ('pred_top1_prob', 'mean'))
                .reset_index())
agg_corr_sev.to_csv(f'{RESULTS_DIR}/agg_corruption_severity.csv', index=False)
print(f'agg_corruption_severity.csv:  {len(agg_corr_sev)} wierszy')

# Tabela 2: per (model, wnid, corruption, severity)
agg_class_corr = (df_all.groupby(['model','wnid','corruption','category','severity'])
                  .agg(n              = ('filename',       'count'),
                       top1_acc       = ('top1_correct',   'mean'),
                       top5_acc       = ('top5_correct',   'mean'),
                       mean_true_prob = ('true_class_prob','mean'))
                  .reset_index())
agg_class_corr.to_csv(f'{RESULTS_DIR}/agg_class_corruption.csv', index=False)
print(f'agg_class_corruption.csv:     {len(agg_class_corr)} wierszy')

# Tabela 3: mPC per (model, wnid) - odporność per klasa
agg_class_mpc = (df_all[df_all['corruption']!='clean']
                 .groupby(['model','wnid'])
                 .agg(mPC_top1  = ('top1_correct',   'mean'),
                      mPC_top5  = ('top5_correct',   'mean'),
                      mean_conf = ('true_class_prob','mean'))
                 .reset_index())
agg_class_mpc.to_csv(f'{RESULTS_DIR}/agg_class_mpc.csv', index=False)
print(f'agg_class_mpc.csv:            {len(agg_class_mpc)} wierszy')

# Tabela 4: mCE per model, kategoria i typ zakłócenia
mce_rows = []
for model_name in MODELS:
    if model_name not in dfs: continue
    sub = df_all[(df_all['model']==model_name) & (df_all['corruption']!='clean')]
    mce_rows.append({'model':model_name,'level':'global','group':'ALL',
                     'top1_acc':float(sub['top1_correct'].mean()),
                     'mCE':float(1-sub['top1_correct'].mean())})
    for cat in CORRUPTIONS:
        s = sub[sub['category']==cat]
        mce_rows.append({'model':model_name,'level':'category','group':cat,
                         'top1_acc':float(s['top1_correct'].mean()),
                         'mCE':float(1-s['top1_correct'].mean())})
    for cname in ALL_CORRUPTIONS:
        s = sub[sub['corruption']==cname]
        mce_rows.append({'model':model_name,'level':'corruption','group':cname,
                         'top1_acc':float(s['top1_correct'].mean()),
                         'mCE':float(1-s['top1_correct'].mean())})

mce_df = pd.DataFrame(mce_rows)
mce_df.to_csv(f'{RESULTS_DIR}/mean_corruption_error.csv', index=False)
print(f'mean_corruption_error.csv:    {len(mce_df)} wierszy')
print()
print('mCE globalne per model:')
print(mce_df[mce_df['level']=='global'][['model','top1_acc','mCE']].to_string(index=False))